# Chess data cleaning for INFO 2950 Phases 3–4
Troy Goetz (tmg85) and Nicholas Grabowski (nrg39)

This notebook preserves the Phase 2 cleaning and sampling rules. It exports the complete prepared dataset and the original 600-game sample separately. Run from this project folder, or open the hosted notebook in Colab. If the local raw files are absent, the code reads the exact archived files from the pinned source commit. The raw archive and lookup are local snapshots from repository commit `79c0e0878c8fe13c2413f0e47cb502393d192f49`; no new API collection is needed.

Install pandas, duckdb, seaborn and matplotlib in your notebook environment if needed.

In [1]:
# Colab users can run this notebook without first downloading the raw files.
from pathlib import Path
import importlib.util, subprocess, sys
missing = [p for p in ['pandas','duckdb','seaborn','matplotlib'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
Path('data').mkdir(exist_ok=True)
SOURCE = 'https://raw.githubusercontent.com/Toyman123/info2950-project/79c0e0878c8fe13c2413f0e47cb502393d192f49/phase2/'


## DATA COLLECTION AND CLEANING

We collected game records from Chess.com's public API. We used the public list of recently active accounts that identified their country as the United States, shuffled the usernames with seed 2950, and selected 300 accounts. We then downloaded each selected account's available archives for July, August, and September 2026. Their opponents could be from any country. The saved archive log records 900 successful monthly requests.

The download contained 64,738 rows. We kept the original CSV and the account-list snapshot so our work can be checked without contacting the API again. The collection notebook is separate; the cells below show the cleaning, grouping, and sampling that turn those records into our analysis dataset.

**Sources and files:** [API documentation](https://www.chess.com/news/view/published-data-api), [raw CSV archive](https://github.com/Toyman123/info2950-project/blob/main/phase2/data/raw/chess_games_raw.csv.gz), [collection records](https://github.com/Toyman123/info2950-project/tree/main/phase2/data/raw), and [opening-label lookup](https://github.com/Toyman123/info2950-project/blob/main/phase2/data/opening_lookup.csv).

### Set up the notebook

We use pandas to read and save tables, DuckDB to filter and summarize them with SQL, and Seaborn to draw the plots. Matplotlib controls titles, axis labels, and spacing. The next cell installs DuckDB if it is missing from the Colab environment.

In [2]:
# Import the packages used in class and choose one plotting style.
import pandas as pd
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
pd.set_option("display.max_columns", 10)
pd.set_option("display.float_format", "{:.2f}".format)

rating_groups = ["Under 1000", "1000–1499", "1500–1999", "2000+"]
rating_colors = ["#33658A", "#D17A22", "#3A8067", "#865D91"]

### Read the source data

The raw CSV includes the full PGN record for each game, which makes it large. PGN is a text format containing game information and moves. The archive is compressed without dropping any rows or columns. We show a few useful columns below instead of printing long move records and URLs across the page.

In [3]:
# Read the archived source tables. pandas decompresses the .gz file.
games = pd.read_csv("data/raw/chess_games_raw.csv.gz" if Path("data/raw/chess_games_raw.csv.gz").exists() else SOURCE + "data/raw/chess_games_raw.csv.gz")
opening_lookup = pd.read_csv("data/opening_lookup.csv" if Path("data/opening_lookup.csv").exists() else SOURCE + "data/opening_lookup.csv")

# Save row order so changing to SQL does not change our existing sample.
games["source_order"] = range(len(games))
audit = [("Original downloaded rows", len(games))]

preview = duckdb.sql("""
    SELECT white_rating, black_rating, time_class,
           time_control, white_result
    FROM games
    ORDER BY source_order
    LIMIT 5
""").df()
display(preview)

,white_rating,black_rating,time_class,time_control,white_result
0,1345,1339,blitz,300,win
1,1331,1343,blitz,300,checkmated
2,1323,1355,blitz,300,resigned
3,1362,1316,blitz,300,win
4,1309,1369,blitz,300,abandoned


### Keep the games that match our question

We keep rated blitz games played under standard chess rules. Blitz is a fast time category; the sample contains three-minute, five-minute, and three-minute-plus-two-second games. We do not combine blitz with bullet, rapid, or chess variants. A game needs a nonblank URL because that URL is how we identify repeated copies of the same game.

In [4]:
# Use SQL WHERE conditions to keep rated standard blitz games.
rated_games = duckdb.sql("""
    SELECT *
    FROM games
    WHERE rated = TRUE AND rules = 'chess' AND time_class = 'blitz'
    ORDER BY source_order
""").df()
audit.append(("Rated standard blitz games", len(rated_games)))

valid_urls = duckdb.sql("""
    SELECT * FROM rated_games
    WHERE game_url IS NOT NULL AND TRIM(game_url) <> ''
    ORDER BY source_order
""").df()
audit.append(("With a nonblank game URL", len(valid_urls)))

# A game can be downloaded through either opponent's archive.
unique_games = valid_urls.drop_duplicates(subset="game_url").copy()
audit.append(("Unique game URLs", len(unique_games)))

### Check ratings and create rating groups

We convert both rating columns to numbers and remove missing or nonpositive ratings. A positive rating gap means White was higher rated; a negative gap means Black was higher rated. For example, White rated 1400 against Black rated 1300 gives a gap of +100.

Our four groups use White's rating at the time of the game. The boundaries are under 1000, 1000–1499, 1500–1999, and 2000 or higher. These are comparison groups we chose, not official skill titles.

In [5]:
# Invalid text becomes missing, so the SQL filter can exclude it.
for column in ["white_rating", "black_rating"]:
    unique_games[column] = pd.to_numeric(
        unique_games[column], errors="coerce"
    )

base = duckdb.sql("""
    SELECT *, white_rating - black_rating AS rating_difference,
           CASE
               WHEN white_rating < 1000 THEN 'Under 1000'
               WHEN white_rating < 1500 THEN '1000–1499'
               WHEN white_rating < 2000 THEN '1500–1999'
               ELSE '2000+'
           END AS rating_group
    FROM unique_games
    WHERE white_rating > 0 AND black_rating > 0
    ORDER BY source_order
""").df()
audit.append(("With valid ratings for both players", len(base)))

### Group openings and record the result

An opening is the sequence of moves at the start of a chess game. An opening family groups related variations under one name. For instance, Alapin and Closed Sicilian labels are grouped with Sicilian. We use Chess.com's opening URLs to make these groups; we do not infer aggression or identify openings by independently replaying the moves.

The lookup file gives every observed opening URL one family label. Its rules keep a name through its first family term, such as “Defense,” “Opening,” or “Gambit,” and combine the documented alternate labels. Giuoco Piano and Pianissimo join Italian; shortened King's Indian and King's Fianchetto labels join their full families; Owens joins Owen's Defense; Magnus Queen's Gambit joins Queen's Gambit. Nimzo-Indian and Bogo-Indian labels keep their family names. “Undefined” is marked “Unknown.” The lookup is a classification of our existing source, not a second independent dataset.

We use a SQL LEFT JOIN to attach these labels without losing a game whose opening is missing. We also read the result from the PGN header and assign White a score. The source result codes include reasons such as resignation or checkmate; the PGN result gives the same win/draw/loss outcome in a consistent form.

In [6]:
# Extract the final result recorded in the PGN header.
base["game_result"] = base["pgn"].str.extract(
    r'\[Result "([^\"]+)"\]', expand=False
)

# Join one family per URL and convert completed results to White scores.
prepared = duckdb.sql("""
    SELECT b.*, COALESCE(o.opening_family, 'Unknown') AS opening_family,
           CASE b.game_result
               WHEN '1-0' THEN 1.0
               WHEN '1/2-1/2' THEN 0.5
               WHEN '0-1' THEN 0.0
           END AS white_score
    FROM base AS b
    LEFT JOIN opening_lookup AS o ON b.opening_url = o.opening_url
    WHERE b.game_result IN ('1-0', '1/2-1/2', '0-1')
    ORDER BY b.source_order
""").df()
audit.append(("With a recognized completed result", len(prepared)))

# Display the cleaning record and save an intermediate file.
cleaning_record = pd.DataFrame(
    audit, columns=["Cleaning step", "Rows remaining"]
)
display(cleaning_record)
prepared.drop(columns="source_order").to_csv(
    "data/chess_games_prepared.csv", index=False
)

,Cleaning step,Rows remaining
0,Original downloaded rows,64738
1,Rated standard blitz games,29385
2,With a nonblank game URL,29385
3,Unique game URLs,29385
4,With valid ratings for both players,29385
5,With a recognized completed result,29385


The filters leave 29,385 games. Checking URLs, duplicate games, ratings, and results did not remove additional rows from this particular download. We show those checks because they are part of the procedure, even when they remove zero games.

### Preserve the original preregistered sample

We selected 150 games at random from each rating group with seed 2950. That gives us 600 games without letting the much larger under-1000 group dominate the comparison. We keep the same game selection used for peer review; changing the code to SQL and updating the presentation does not create a new sample. We use pandas for this sampling step and SQL for the summaries below.

In [7]:
# Select the same 150 games from each group, using the original row order.
samples = []
for group in rating_groups:
    group_games = prepared[prepared["rating_group"] == group]
    if len(group_games) < 150:
        raise ValueError("A rating group has fewer than 150 games.")
    samples.append(group_games.sample(n=150, random_state=2950))

analysis_games = pd.concat(samples, ignore_index=True)
analysis_games = analysis_games.sample(
    frac=1, random_state=2950
).reset_index(drop=True)
analysis_games = analysis_games.drop(columns="source_order")

# Save the analysis-ready data, then check the sample size with SQL.
analysis_games.to_csv("data/chess_analysis_600_reproduced.csv", index=False)
sample_check = duckdb.sql("""
    SELECT rating_group AS "White rating group", COUNT(*) AS "Games",
           COUNT(DISTINCT game_url) AS "Unique games",
           COUNT(DISTINCT sampled_player) AS "Archive accounts"
    FROM analysis_games
    GROUP BY rating_group
""").df()
sample_check = sample_check.set_index("White rating group").reindex(
    rating_groups
).reset_index()
display(sample_check)

,White rating group,Games,Unique games,Archive accounts
0,Under 1000,150,150,45
1,1000–1499,150,150,10
2,1500–1999,150,150,3
3,2000+,150,150,3


### Reproduce the archived sample and export model-ready columns
The CSV found in Downloads is the archived preregistered sample. Check that regeneration gives the same game IDs, ratings, groups and scores. The Downloads CSV uses an older collapsed opening classification (many labels are Other opening); use the revised notebook’s lookup to regenerate family labels for both datasets. Preserve the old CSV only as a historical source. Add numerical model indicators here so the report only imports analysis-ready data. Complete-case and deduplication checks remain in this cleaning notebook.

The model-ready CSVs omit only the long PGN text to keep the report package small. Raw and unaugmented prepared files retain it. This does not change any modeled value, game inclusion or family assignment.

In [8]:
from pathlib import Path
archived = pd.read_csv('data/chess_analysis_600.csv' if Path('data/chess_analysis_600.csv').exists() else SOURCE + 'data/chess_analysis_600.csv')
assert set(archived.game_url) == set(analysis_games.game_url)
check_cols = ['white_rating','black_rating','rating_difference','rating_group','white_score']
pd.testing.assert_frame_equal(archived.set_index('game_url')[check_cols].sort_index(), analysis_games.set_index('game_url')[check_cols].sort_index(), check_dtype=False)
for frame, name in [(prepared.drop(columns='source_order'), 'chess_games_prepared'), (analysis_games, 'chess_analysis_600')]:
    frame = frame.copy()
    frame['gap100'] = frame['rating_difference'] / 100
    frame['sicilian'] = (frame['opening_family'] == 'Sicilian').astype(int)
    frame['kings_pawn'] = (frame['opening_family'] == "King's Pawn Opening").astype(int)
    frame['rating_1000_1499'] = ((frame['white_rating'] >= 1000) & (frame['white_rating'] < 1500)).astype(int)
    assert frame.game_url.is_unique
    assert frame.white_score.isin([0, 0.5, 1]).all()
    assert not frame[check_cols].isna().any().any()
    # Keep full PGN in raw/prepared files; models do not need the long move text.
    frame = frame.drop(columns=['pgn'])
    frame.to_csv('data/' + name + '_model_ready.csv', index=False)
print('Game IDs, ratings, groups and scores reproduced; opening labels rebuilt using revised lookup; full prepared games:', len(prepared))
print('Opening labels:', sorted(prepared.opening_family.unique())[:20])

Game IDs, ratings, groups and scores reproduced; opening labels rebuilt using revised lookup; full prepared games: 29385
Opening labels: ["Alekhine's Defense", 'Anderssen Opening', 'Benko Gambit', 'Benoni Defense', "Bird's Opening", "Bishop's Opening", 'Bogo-Indian Defense', 'Budapest Gambit', 'Caro Kann', 'Caro-Kann', 'Catalan Opening', 'Center Game', 'Clemenz Opening', 'Colle', 'Colle System', 'Danish Gambit', 'Dutch', 'Dutch Defense', 'English', 'English Defense']
